# Explore Chroma Vector Store
Inspect chunks, metadata, and run similarity searches against the local DB.

In [ ]:
import sys
sys.path.insert(0, '..')  # make app/ importable

import chromadb

client = chromadb.PersistentClient(path='../chroma_db')
col = client.get_collection('bayer-rag')

print('Collections:', client.list_collections())
print('Total chunks:', col.count())

: 

## Peek at first 5 chunks

In [ ]:
results = col.peek(5)

for i, (doc_id, doc, meta) in enumerate(zip(results['ids'], results['documents'], results['metadatas'])):
    print(f'--- Chunk {i} ---')
    print(f'ID:     {doc_id}')
    print(f'Source: {meta.get("source", "unknown")}')
    print(f'Text:   {doc[:300]}')
    print()

## All chunks grouped by source file

In [ ]:
from collections import Counter

all_meta = col.get(include=['metadatas'])['metadatas']
sources = Counter(m.get('source', 'unknown') for m in all_meta)

print(f'{'Source':<60} Chunks')
print('-' * 70)
for source, count in sorted(sources.items()):
    print(f'{source:<60} {count}')

## Similarity search (no LLM — raw vector retrieval)

In [ ]:
from dotenv import load_dotenv
load_dotenv('../.env')

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=os.environ['OPENAI_API_KEY'],
)

vector_store = Chroma(
    client=client,
    collection_name='bayer-rag',
    embedding_function=embeddings,
)

QUERY = 'What are the active ingredients?'

docs = vector_store.similarity_search_with_score(QUERY, k=4)

for doc, score in docs:
    print(f'Score (distance): {score:.4f}')
    print(f'Source: {doc.metadata.get("source")}')
    print(f'Text:   {doc.page_content[:300]}')
    print()

## Peek at raw vector dimensions

In [ ]:
result = col.get(include=['embeddings'], limit=1)
vector = result['embeddings'][0]

print(f'Vector dimensions: {len(vector)}')
print(f'First 10 values:   {[round(v, 6) for v in vector[:10]]}')